In [3]:
import cv2
import numpy as np
from IPython.display import Video, display


input_path = "/Users/shivam/Downloads/Volleyball.mp4"
output_path = "/Users/shivam/Downloads/volleyball_tracked.mp4"

# Open video
cap = cv2.VideoCapture(input_path)

In [4]:
if not cap.isOpened():
    print(f"Error: Could not open video at {input_path}")
else:
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Create the output video 
    fourcc = cv2.VideoWriter_fourcc(*"avc1")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

   
    lower = np.array([10, 100, 100])
    upper = np.array([35, 255, 255])

    kernel = np.ones((3, 3), np.uint8)
    previous = None
    path = []
    missed = 0
    detected = 0

In [8]:
while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Select pixels matching the balls colour
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, lower, upper)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        # outlines of the selected regions
        contours, _ = cv2.findContours(
            mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        best = None
        best_distance = 1000000

        for contour in contours:
            area = cv2.contourArea(contour)
            if area < 20 or area > 250:
                continue

            (x, y), radius = cv2.minEnclosingCircle(contour)
            if radius < 3 or radius > 12 or y < 145:
                continue

            x, y, radius = int(x), int(y), int(radius)

            if previous is None:
               
                if y < 260 and (best is None or area > best[3]):
                    best = (x, y, radius, area)
            else:
              
                distance = (x - previous[0]) ** 2 + (y - previous[1]) ** 2
                if distance < best_distance and distance < 10000:
                    best_distance = distance
                    best = (x, y, radius, area)

        if best is not None:
            x, y, radius, area = best
            previous = (x, y)
            missed = 0
            detected += 1
            path.append((x, y))

            # Keep only the last 25 positions
            if len(path) > 25:
                path.pop(0)

            # Circle around the current position
            cv2.circle(frame, (x, y), radius, (0, 255, 0), 2)

            # Join the recent positions to show the path
            for i in range(1, len(path)):
                cv2.line(frame, path[i - 1], path[i], (0, 255, 255), 2)
        else:
            missed += 1
            if missed > 6:
                previous = None
                path = []

        out.write(frame)

    cap.release()
    out.release()
    cv2.destroyAllWindows()

    print(f"Done! Saved to: {output_path}")
    print("Frames with a detected ball candidate:", detected)

    # Display the video player directly inside Jupyter
    display(Video(output_path, embed=True))

IndentationError: unindent does not match any outer indentation level (<string>, line 66)